In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [2]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()
    
    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()
    
    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [7]:
# MAIN (DEBUG)
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 4
print("Asset class:", asset_classes[asset_index])
# Load data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-10-31')

#Display
# print (asset_classes[asset_index],":\n")
# print(data)
with pd.option_context('display.max_columns', None,
                       'display.max_rows', None,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
    # display(data.head(15))

Asset class: US Treasuries
